In [21]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [22]:
# hyperparameters
batch_size = 1          # how many independent sequences will we process in parallel?
block_size = 256        # what is the maximum context length for predictions?
n_embd = 384            # size of embedding vectors for each token
eval_iters = 200        # number of iterations to use when estimation valuation loss
learning_rate = 3e-4    # learning rate for the optimizer to use
max_iters = 5000        # iterations to use for training the model against each batch
eval_interval = 500     # how often to show the training and valuation loss estimates
n_head = 6              # the number of simulataneous self attention heads
n_layer = 6             # number of feed forward layers in the model
dropout = 0.2
# ------------

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

In [23]:
# Read it in to inspect it
with open("wonderland.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [24]:
print("Length of dataset in characters: ", len(text))

Length of dataset in characters:  144009


In [25]:
# Look at the first 100 characters
print(text[:100])

CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on t


In [26]:
# get all the unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"'(),-.:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz
68


In [27]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
# encoder: takes a string and outputs a list of integers
encode = lambda s: [stoi[c] for c in s]
# decoder: takes a list of integers and outputs a string
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("Hi there"))
print(decode(encode("Hi there")))

[20, 50, 1, 61, 49, 46, 59, 46]
Hi there


In [28]:
# lets now encode the entire text dataset and store it in a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:100]) # the first 100 characters

torch.Size([144009]) <built-in method type of Tensor object at 0xffff18162710>
tensor([15, 20, 13, 28, 32, 17, 30,  1, 21,  9,  1, 16, 56, 64, 55,  1, 61, 49,
        46,  1, 30, 42, 43, 43, 50, 61,  8, 20, 56, 53, 46,  0,  0, 13, 53, 50,
        44, 46,  1, 64, 42, 60,  1, 43, 46, 48, 50, 55, 55, 50, 55, 48,  1, 61,
        56,  1, 48, 46, 61,  1, 63, 46, 59, 66,  1, 61, 50, 59, 46, 45,  1, 56,
        47,  1, 60, 50, 61, 61, 50, 55, 48,  1, 43, 66,  1, 49, 46, 59,  1, 60,
        50, 60, 61, 46, 59,  1, 56, 55,  1, 61])


In [29]:
# split data into training and validation sets
# use the first 90% for training and the remaining 10% for validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [30]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)

    return x, y

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)

        return out

class Attention2(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x):
        B, T, C = x.shape
        tril = torch.tril(torch.ones(T, T, device=device))
        scores = x @ x.transpose(-2, -1)
        scores = scores.masked_fill(tril == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = weights @ x

        return out

class Attention3(nn.Module):

    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_embd, n_embd))

    def forward(self, x):
        B, T, C = x.shape
        tril = torch.tril(torch.ones(T, T, device=device))
        scores = x @ self.weight @ x.transpose(-2, -1)
        scores = scores.masked_fill(tril == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = weights @ x

        return out

class NaiveHead(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.weight = nn.Parameter(
            torch.randn(block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape

        tril = torch.tril(
            torch.ones(T, T, device=x.device)
        )

        wei = self.weight[:T, :T]
        wei = wei.masked_fill(
            tril == 0,
            float("-inf")
        )
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)

        return wei @ v

class NaiveHead1(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.register_buffer('tril', torch.tril(torch.ones(n_embd, n_embd)))
        self.weight = nn.Parameter(torch.randn(block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        wei = self.weight[:T, :T]
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=1)
        out = wei @ x

        return out

class MultiHeadNaiveAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        head_size = n_embd // num_heads

        self.heads = nn.ModuleList(
            [NaiveHead(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        out = self.dropout(self.proj(out))

        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

class LanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                               # (B,T,n_emdd)      - (1,8,8)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb                                                   # (B,T,C)
        x = self.blocks(x)                                                      # (B,T,n_embd)      - (1,8,8)
        x = self.ln_f(x)                                                        # (B,T,n_embd)      - (1,8,8)
        logits = self.lm_head(x)                                                # (B,T,vocab_size)  - (1,8,68)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape                          # (1,8,68)
            logits = logits.view(B*T, C)                    # (8,68)
            targets = targets.view(B*T)                     # (8,1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [32]:
model = LanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.791236 M parameters


In [33]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [34]:
def generate_output():

    prompt = " "

    context = torch.tensor(
        encode(prompt),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

In [35]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("**** OUTPUT ****")
generate_output()
print("****.       ****")

AttributeError: 'Head' object has no attribute 'dropout'